# MCS603 — Data Analysis Module
## Notebook 1: SciPy — Normality Tests, IQR, Winsorization, t-Tests & Outlier Detection

---
### Learning Objectives
- Test whether data follows a normal distribution (Shapiro-Wilk, D'Agostino, Anderson-Darling)
- Detect outliers using IQR, Z-score, and Modified Z-score methods
- Apply Winsorization to cap extreme values
- Run one-sample, two-sample (independent), and paired t-tests
- Interpret p-values and draw statistical conclusions

### Setup
```bash
pip install scipy numpy pandas matplotlib
```

### Outline
1. Dataset — Africa Health Indicators  
2. Normality Tests  
3. Q-Q Plots  
4. Outlier Detection (IQR & Z-score)  
5. Winsorization  
6. t-Tests  
7. Exercises  

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy import stats
from scipy.stats.mstats import winsorize

np.random.seed(42)
plt.rcParams.update({"figure.dpi": 120, "axes.grid": True,
                     "grid.alpha": 0.3, "figure.figsize": (10, 4)})

---
## 1. Dataset — Africa Health Indicators

We use a synthetic dataset of **54 African countries** with realistic health and economic indicators.  
The same dataset is used throughout all five notebooks.

In [ ]:
np.random.seed(42)
N = 54   # African countries

countries = [
    "Nigeria","Ethiopia","Egypt","DR Congo","Tanzania","South Africa","Kenya",
    "Uganda","Algeria","Sudan","Morocco","Mozambique","Ghana","Angola","Cameroon",
    "Madagascar","Côte d'Ivoire","Niger","Burkina Faso","Mali","Malawi","Zambia",
    "Senegal","Chad","Somalia","Zimbabwe","Guinea","Rwanda","Benin","Burundi",
    "Tunisia","South Sudan","Togo","Sierra Leone","Libya","Congo","Liberia",
    "Mauritania","Eritrea","Namibia","Gambia","Botswana","Gabon","Lesotho",
    "Guinea-Bissau","Equatorial Guinea","Mauritius","Eswatini","Djibouti",
    "Comoros","Cabo Verde","Sao Tome","Seychelles","São Tomé"
]

regions = (
    ["West Africa"]*16 + ["East Africa"]*14 +
    ["North Africa"]*6 + ["Central Africa"]*8 + ["Southern Africa"]*10
)

df = pd.DataFrame({
    "country"           : countries,
    "region"            : regions,
    "year"              : 2022,
    # Life expectancy: roughly normal, mean ~63, sd ~8
    "life_expectancy"   : np.clip(np.random.normal(63, 8, N), 45, 80),
    # Infant mortality (per 1000): right-skewed
    "infant_mortality"  : np.clip(np.random.exponential(35, N), 5, 120),
    # Maternal mortality (per 100k): right-skewed with outliers
    "maternal_mortality": np.clip(np.random.exponential(350, N) +
                                  np.random.choice([0]*50 + [800, 900, 1100, 1200], N), 20, 2000),
    # HIV prevalence (%): right-skewed
    "hiv_prevalence"    : np.clip(np.random.exponential(3.5, N), 0.1, 28),
    # Healthcare expenditure (% GDP)
    "health_expenditure": np.clip(np.random.normal(5.2, 2.1, N), 1, 12),
    # GDP per capita (USD): log-normal
    "gdp_per_capita"    : np.clip(np.exp(np.random.normal(7.2, 1.2, N)), 300, 15000),
    # Vaccination coverage DTP3 (%)
    "vaccination_dtp3"  : np.clip(np.random.normal(78, 18, N), 20, 99),
    # Access to clean water (%)
    "water_access"      : np.clip(np.random.normal(68, 22, N), 15, 99),
    # Malaria incidence per 1000
    "malaria_incidence" : np.clip(np.random.exponential(120, N), 0, 450),
    # Physicians per 1000 people
    "physicians_per1k"  : np.clip(np.random.exponential(0.8, N), 0.02, 5),
})

# Introduce realistic NaNs
for col in ["maternal_mortality", "physicians_per1k", "malaria_incidence"]:
    mask = np.random.choice([True, False], N, p=[0.12, 0.88])
    df.loc[mask, col] = np.nan

df.to_csv("africa_health.csv", index=False)
print(df.shape)
df.head()

---
## 2. Normality Tests

Many statistical procedures assume normality. We test this formally before applying them.

| Test | Function | Best for |
|---|---|---|
| Shapiro-Wilk | `stats.shapiro` | Small to medium samples (n < 2000) |
| D'Agostino-Pearson | `stats.normaltest` | Tests skewness + kurtosis |
| Anderson-Darling | `stats.anderson` | More sensitive in tails |

**Null hypothesis (H₀):** the data are normally distributed.  
If **p < 0.05**, reject H₀ — data are likely NOT normal.

In [ ]:
def normality_report(series, name):
    data = series.dropna().values
    print(f"\n{'='*50}")
    print(f"Normality Tests: {name}  (n={len(data)})")
    print(f"{'='*50}")
    print(f"  Mean   : {data.mean():.3f}")
    print(f"  Std    : {data.std():.3f}")
    print(f"  Skew   : {stats.skew(data):.3f}")
    print(f"  Kurtosis: {stats.kurtosis(data):.3f}")

    # Shapiro-Wilk
    sw_stat, sw_p = stats.shapiro(data)
    print(f"\n  Shapiro-Wilk:        W={sw_stat:.4f}  p={sw_p:.4f}  "
          f"{'NORMAL' if sw_p > 0.05 else 'NOT NORMAL'}")

    # D'Agostino-Pearson
    dp_stat, dp_p = stats.normaltest(data)
    print(f"  D'Agostino-Pearson:  K²={dp_stat:.4f}  p={dp_p:.4f}  "
          f"{'NORMAL' if dp_p > 0.05 else 'NOT NORMAL'}")

    # Anderson-Darling
    ad = stats.anderson(data, dist="norm")
    ad_result = "NORMAL" if ad.statistic < ad.critical_values[2] else "NOT NORMAL"
    print(f"  Anderson-Darling:    A²={ad.statistic:.4f}  "
          f"(5% critical={ad.critical_values[2]:.4f})  {ad_result}")

columns_to_test = ["life_expectancy", "infant_mortality", "maternal_mortality",
                   "gdp_per_capita", "health_expenditure"]

for col in columns_to_test:
    normality_report(df[col], col)

---
## 3. Q-Q Plots

A **Q-Q (Quantile-Quantile) plot** visually checks normality.  
If data are normal, points fall along the diagonal reference line.  
Deviations at the tails indicate skewness or heavy tails.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

plot_cols = ["life_expectancy", "infant_mortality", "maternal_mortality",
             "gdp_per_capita", "health_expenditure", "hiv_prevalence"]

for ax, col in zip(axes, plot_cols):
    data = df[col].dropna()
    stats.probplot(data, dist="norm", plot=ax)
    ax.set_title(f"Q-Q: {col.replace('_', ' ').title()}", fontsize=10)
    ax.get_lines()[0].set(markersize=4, alpha=0.6)

plt.suptitle("Q-Q Plots — Africa Health Indicators", fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig("qq_plots.png", bbox_inches="tight")
plt.show()

---
## 4. Outlier Detection

### 4.1 IQR Method

The **Interquartile Range (IQR)** = Q3 − Q1 (the middle 50% of data).

```
Lower fence = Q1 − 1.5 × IQR
Upper fence = Q3 + 1.5 × IQR
```

Values outside these fences are flagged as outliers. Use **3.0 × IQR** for extreme outliers.

In [ ]:
def iqr_outliers(series, k=1.5):
    """Return outlier mask using IQR method."""
    q1  = series.quantile(0.25)
    q3  = series.quantile(0.75)
    iqr = q3 - q1
    lo  = q1 - k * iqr
    hi  = q3 + k * iqr
    mask = (series < lo) | (series > hi)
    return mask, lo, hi

print(f"{'Column':<25} {'Q1':>8} {'Q3':>8} {'IQR':>8} "
      f"{'Lower':>8} {'Upper':>8} {'Outliers':>10}")
print("-" * 82)

outlier_summary = {}
for col in ["infant_mortality", "maternal_mortality", "hiv_prevalence",
            "gdp_per_capita", "malaria_incidence"]:
    s = df[col].dropna()
    mask, lo, hi = iqr_outliers(s)
    outlier_summary[col] = df[col].index[df[col].notna()][mask]
    print(f"{col:<25} {s.quantile(.25):>8.1f} {s.quantile(.75):>8.1f} "
          f"{s.quantile(.75)-s.quantile(.25):>8.1f} "
          f"{lo:>8.1f} {hi:>8.1f} {mask.sum():>10}")

In [ ]:
# Visual: box plots with outliers highlighted
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
plot_targets = [("infant_mortality", "Infant Mortality (per 1000)"),
                ("maternal_mortality", "Maternal Mortality (per 100k)"),
                ("gdp_per_capita", "GDP per Capita (USD)")]

for ax, (col, label) in zip(axes, plot_targets):
    data = df[col].dropna()
    mask, lo, hi = iqr_outliers(data)
    normal  = data[~mask]
    outliers = data[mask]

    ax.boxplot(data, vert=True, patch_artist=True,
               boxprops=dict(facecolor="#AED6F1"),
               medianprops=dict(color="navy", linewidth=2),
               flierprops=dict(marker="o", color="crimson", markersize=6))
    ax.set_title(f"{label}\n({len(outliers)} outliers)", fontsize=10)
    ax.set_xticks([])

plt.suptitle("IQR Outlier Detection", fontsize=13)
plt.tight_layout()
plt.savefig("iqr_boxplots.png", bbox_inches="tight")
plt.show()

### 4.2 Z-Score & Modified Z-Score

| Method | Formula | Threshold | Notes |
|---|---|---|---|
| Z-score | `(x - mean) / std` | \|z\| > 3 | Assumes normality |
| Modified Z-score | `0.6745 × (x - median) / MAD` | \|mz\| > 3.5 | Robust to non-normal data |

In [ ]:
def zscore_outliers(series, threshold=3.0):
    z = np.abs(stats.zscore(series.dropna()))
    return z > threshold

def modified_zscore_outliers(series, threshold=3.5):
    data   = series.dropna()
    median = data.median()
    mad    = np.median(np.abs(data - median))
    mz     = 0.6745 * np.abs(data - median) / (mad + 1e-9)
    return mz > threshold

# Compare all three methods on maternal_mortality
col  = "maternal_mortality"
data = df[col].dropna()

iqr_mask  = iqr_outliers(data)[0]
z_mask    = zscore_outliers(df[col])
mz_mask   = modified_zscore_outliers(df[col])

print(f"Outlier comparison for: {col}")
print(f"  IQR method          : {iqr_mask.sum()} outliers")
print(f"  Z-score (|z|>3)     : {z_mask.sum()} outliers")
print(f"  Modified Z (|mz|>3.5): {mz_mask.sum()} outliers")

print("\nCountries flagged by Modified Z-score:")
flagged_idx = df[col].dropna().index[mz_mask]
print(df.loc[flagged_idx, ["country", col]].to_string(index=False))

---
## 5. Winsorization

**Winsorization** replaces extreme values with the value at a given percentile — it preserves sample size unlike removing outliers.

```
limits=(0.05, 0.05) means:
  bottom 5% → replaced with the 5th percentile value
  top    5% → replaced with the 95th percentile value
```

In [ ]:
col  = "maternal_mortality"
raw  = df[col].dropna().values

# Apply Winsorization at 5th and 95th percentiles
w05  = winsorize(raw, limits=[0.05, 0.05])
w10  = winsorize(raw, limits=[0.10, 0.10])

print(f"{'Statistic':<12} {'Original':>12} {'Win. 5%':>12} {'Win. 10%':>12}")
print("-" * 52)
for label, arr in [("Original", raw), ("Win. 5%", w05), ("Win. 10%", w10)]:
    print(f"{label:<12} {arr.mean():>12.1f} {np.std(arr):>12.1f} "
          f"{arr.max():>12.1f}")

# Before/After visualisation
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=False)
for ax, (label, arr, color) in zip(
    axes,
    [("Original", raw, "#E74C3C"),
     ("Winsorized 5%", np.array(w05), "#F39C12"),
     ("Winsorized 10%", np.array(w10), "#27AE60")]
):
    ax.hist(arr, bins=20, color=color, alpha=0.75, edgecolor="white")
    ax.axvline(np.mean(arr), color="black", linestyle="--", label=f"mean={np.mean(arr):.0f}")
    ax.set_title(label, fontsize=11)
    ax.legend(fontsize=8)
    ax.set_xlabel("Maternal Mortality")

plt.suptitle("Effect of Winsorization on Maternal Mortality Distribution", fontsize=12)
plt.tight_layout()
plt.savefig("winsorization.png", bbox_inches="tight")
plt.show()

---
## 6. t-Tests

| Test | Question | Function |
|---|---|---|
| One-sample t | Is the mean equal to a known value? | `stats.ttest_1samp` |
| Independent two-sample t | Are two group means equal? | `stats.ttest_ind` |
| Welch's t | Two groups, unequal variance | `stats.ttest_ind(equal_var=False)` |
| Paired t | Before/after on the same subjects | `stats.ttest_rel` |

**Decision rule:** if p-value < α (usually 0.05) → reject H₀

In [ ]:
def ttest_report(name, stat, p, alpha=0.05):
    verdict = "REJECT H₀" if p < alpha else "FAIL TO REJECT H₀"
    print(f"  {name:<35}  t={stat:>8.4f}  p={p:.4f}  → {verdict}")

In [ ]:
# One-sample t-test
# H₀: Mean life expectancy in Africa = 65 years (WHO target)
le = df["life_expectancy"].dropna()
t_stat, p_val = stats.ttest_1samp(le, popmean=65)

print("One-Sample t-Test")
print(f"  H₀: Mean life expectancy = 65 years")
print(f"  Sample mean = {le.mean():.2f}, n = {len(le)}")
ttest_report("Life Expectancy vs. 65", t_stat, p_val)

# 95% Confidence Interval
ci = stats.t.interval(0.95, df=len(le)-1, loc=le.mean(), scale=stats.sem(le))
print(f"  95% CI: [{ci[0]:.2f}, {ci[1]:.2f}]")

In [ ]:
# Independent two-sample t-test
# H₀: Mean life expectancy is the same in North Africa vs Sub-Saharan Africa
north    = df[df["region"] == "North Africa"]["life_expectancy"].dropna()
sub_sah  = df[df["region"] != "North Africa"]["life_expectancy"].dropna()

print("\nIndependent Two-Sample t-Test")
print(f"  North Africa mean   : {north.mean():.2f}  (n={len(north)})")
print(f"  Sub-Saharan mean    : {sub_sah.mean():.2f}  (n={len(sub_sah)})")
print(f"  H₀: means are equal")

# Equal variance
t, p = stats.ttest_ind(north, sub_sah)
ttest_report("Equal variance (Student's)", t, p)

# Unequal variance (Welch's — default, more robust)
t, p = stats.ttest_ind(north, sub_sah, equal_var=False)
ttest_report("Unequal variance (Welch's)", t, p)

# Levene's test to check variance equality
lev_stat, lev_p = stats.levene(north, sub_sah)
print(f"\n  Levene's variance test: F={lev_stat:.4f}  p={lev_p:.4f}  "
      f"→ variances {'EQUAL' if lev_p > 0.05 else 'UNEQUAL'} at α=0.05")

In [ ]:
# Paired t-test: simulated intervention effect
# Scenario: life expectancy before and after a 5-year health program
np.random.seed(7)
n      = 30
before = np.clip(np.random.normal(58, 7, n), 40, 75)
effect = np.clip(np.random.normal(3.2, 2.1, n), -1, 9)   # intervention effect
after  = before + effect

t_stat, p_val = stats.ttest_rel(before, after)
print("\nPaired t-Test — Health Intervention Effect")
print(f"  H₀: No difference before/after intervention")
print(f"  Before: mean={before.mean():.2f}  After: mean={after.mean():.2f}")
print(f"  Mean improvement: {(after - before).mean():.2f} years")
ttest_report("Paired t (before vs. after)", t_stat, p_val)

# Effect size — Cohen's d
diff   = after - before
cohens_d = diff.mean() / diff.std()
print(f"  Cohen's d = {cohens_d:.3f}  "
      f"({'small' if abs(cohens_d) < 0.5 else 'medium' if abs(cohens_d) < 0.8 else 'large'} effect)")

In [ ]:
# One-Way ANOVA: life expectancy across all 5 regions
groups = [df[df["region"] == r]["life_expectancy"].dropna().values
          for r in df["region"].unique()]

f_stat, p_val = stats.f_oneway(*groups)

print("\nOne-Way ANOVA — Life Expectancy by Region")
print(f"  H₀: All regional means are equal")
for r, g in zip(df["region"].unique(), groups):
    print(f"  {r:<20}: mean={g.mean():.1f}  n={len(g)}")
print(f"\n  F = {f_stat:.4f},  p = {p_val:.4f}")
print(f"  → {'REJECT H₀ — regions differ' if p_val < 0.05 else 'FAIL TO REJECT H₀'}")

---
## 7. Exercises

### Exercise 1
Test whether `vaccination_dtp3` is normally distributed using all three tests. What does the Q-Q plot show?

In [ ]:
# Exercise 1


### Exercise 2
Using the IQR method, identify outlier countries in `hiv_prevalence`. Which countries are flagged? Winsorize the column at 5% and compare the mean before and after.

In [ ]:
# Exercise 2


### Exercise 3
Test whether East African countries have significantly different `infant_mortality` compared to Southern African countries using Welch's t-test. Report the test statistic, p-value, and Cohen's d.

In [ ]:
# Exercise 3


---
## Summary

| Concept | Function | Key Insight |
|---|---|---|
| Shapiro-Wilk | `stats.shapiro` | Best for n < 2000; p < 0.05 → not normal |
| D'Agostino | `stats.normaltest` | Tests skewness & kurtosis jointly |
| Anderson-Darling | `stats.anderson` | Compare statistic to critical value |
| IQR outliers | Manual with `quantile()` | Robust fence = Q1/Q3 ± 1.5×IQR |
| Modified Z | Manual | Better than Z-score for skewed data |
| Winsorize | `mstats.winsorize` | Keeps sample size, caps extremes |
| One-sample t | `stats.ttest_1samp` | Compare sample mean to known value |
| Two-sample t | `stats.ttest_ind` | Use `equal_var=False` by default |
| Paired t | `stats.ttest_rel` | Before/after on same subjects |

**Next:** Notebook 2 — Pandas Data Wrangling & Feature Engineering